In [1]:
import xarray as xr
import numpy as np
from scipy.interpolate import griddata
import sys
import os   

sys.path.append(os.path.join(os.getcwd(), '..'))
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import r2_score


In [2]:
def load_and_process_nc_files(year: int):
    nc_file_buoy = f'../data/years_dataframe/{year}_data.nc'
    nc_file_era5 = f'../ERA5/Processed/era5_{year}_-83_-65_37_49.nc'
    ds_buoy = xr.open_dataset(nc_file_buoy)
    ds_era5 = xr.open_dataset(nc_file_era5)
    ds_era5_time_steps = ds_era5['time'].values
    ds_buoys_time_steps = ds_buoy['time'].values

    # Find common time steps
    common_time_steps = np.intersect1d(ds_era5_time_steps, ds_buoys_time_steps)


    ds_era5_filtered = ds_era5.sel(time=common_time_steps)
    ds_buoy_filtered = ds_buoy.sel(time=common_time_steps)
    
    return ds_era5_filtered, ds_buoy_filtered

In [3]:
def select_and_sort_stations(ds: xr.Dataset):
    stations = ds['station_id'].values[:2]
    return stations

In [4]:
def find_closest_points(station_coords: np.array,  latitudes: np.array, longitudes: np.array, ds: xr.Dataset):
    # Calculate absolute differences from the target
    differences_lat = np.abs(latitudes - station_coords[0]) 

    # Get indices of the two smallest differences
    closest_indices_lat = np.argsort(differences_lat)[:2]

    # Retrieve the two closest values
    closest_values_lat = latitudes[closest_indices_lat]

    difference_long = np.abs(longitudes - station_coords[1])
    closest_indices_long = np.argsort(difference_long)[:2]
    closest_values_long = longitudes[closest_indices_long]
    points = np.array([[closest_values_lat[0], closest_values_long[0]],
                        [closest_values_lat[0], closest_values_long[1]],
                        [closest_values_lat[1], closest_values_long[0]],
                        [closest_values_lat[1], closest_values_long[1]]])
    interp_point = np.array([[station_coords[0], station_coords[1]]])
    values = ds.sel(longitude=closest_values_long, latitude=closest_values_lat)
    
    return points, interp_point, values

In [5]:
def calculate_interpolate_values(points: np.array, interp_point: np.array, ds_buoy_filtered: xr.Dataset, station: str, values: xr.Dataset):
    
    def interpolate_column(col):
            return griddata(points, col, interp_point, method='linear')
        
    variables_to_interpolate = ['u10', 'v10', 'ssr', 't2m', 'd2m']  
    interpolated_values = {}
    

    # Loop over the variables
    for variable in variables_to_interpolate:
        values_variable = values[variable].values
        N = values_variable.shape[0]
        values_variables_reshaped = values_variable.reshape(N, 4)
    
        interpolated_values = np.apply_along_axis(interpolate_column, 1, values_variables_reshaped)
        
        
        
        ds_buoy_filtered[variable + '_interp'] = (('time', station), interpolated_values)
    
    return ds_buoy_filtered

In [6]:
def post_process_before_corr(ds_buoy_filtered: xr.Dataset, station: str, variable: str):
    
    if variable == 'u':
        
        buoy_data = np.array(ds_buoy_filtered.sel(station_id=station)['u_velocity'])
        interp_data = np.array(ds_buoy_filtered.sel(station_id=station)['u10_interp'])
    elif variable == 'v':
        buoy_data = np.array(ds_buoy_filtered.sel(station_id=station)['v_velocity'])
        interp_data = np.array(ds_buoy_filtered.sel(station_id=station)['v10_interp'])
    elif variable == 't':
        buoy_data = np.array(ds_buoy_filtered.sel(station_id=station)['ATMP'])
        interp_data = np.array(ds_buoy_filtered.sel(station_id=station)['t2m_interp'])
    else:
        raise ValueError('Invalid correlation type')
    
    nan_idx = np.isnan(buoy_data)
    buoy_data = buoy_data[~nan_idx]
    interp_data = interp_data[~nan_idx]
    
    return buoy_data, interp_data
    
    

In [7]:
years = [2019]
correlation_u = {'2019': [], '2020': [], '2021': [], '2022': [], '2023': []}
correlation_v = {'2019': [], '2020': [], '2021': [], '2022': [], '2023': []}
correlation_t2m = {'2019': [], '2020': [], '2021': [], '2022': [], '2023': []}
ds_buoy_filtered_dict = {'2019': xr.Dataset, '2020': xr.Dataset, '2021': xr.Dataset, '2022': xr.Dataset, '2023': xr.Dataset}
ds_era5_filtered_dict = {'2019': xr.Dataset, '2020': xr.Dataset, '2021': xr.Dataset, '2022': xr.Dataset, '2023': xr.Dataset} 
stations_dict = {'2019': [], '2020': [], '2021': [], '2022': [], '2023': []} 


In [8]:
for year in years:
    ds_era5_filtered, ds_buoy_filtered = load_and_process_nc_files(year)
    latitudes = np.array(ds_era5_filtered['latitude'])
    longitudes = np.array(ds_era5_filtered['longitude'])
    stations = select_and_sort_stations(ds_buoy_filtered)
    
    for i, station in enumerate(stations):
        
        station_coords = [float(ds_buoy_filtered.sel(station_id=station).latitude.values), float(ds_buoy_filtered.sel(station_id=station).longitude.values)] 

        points, interp_point, values = find_closest_points(station_coords, latitudes, longitudes, ds_era5_filtered)
        
        
        ds_buoy_filtered = calculate_interpolate_values(points, interp_point, ds_buoy_filtered, station, values)
        
    ds_era5_filtered_dict[str(year)], ds_buoy_filtered_dict[str(year)] = ds_era5_filtered, ds_buoy_filtered
    stations_dict[str(year)] = stations
    
    
    
    


In [9]:
def calculate_proximity(proximity_type: str, buoy_data: np.array, interpolated_data: np.array, ax_: plt.Axes ):
    if proximity_type == 'correlation':
        value = np.corrcoef(buoy_data, buoy_data[:, 0])[0, 1]
        sns.regplot(x=buoy_data,y=interpolated_data , ax=ax_)
        ax_.set_title(f'Correlation for {station} with a correlation: {value:.2f}')
        
    elif proximity_type == 'time_series':
        ax_.plot(buoy_data, label='Buoy Data')
        ax_.plot(interpolated_data, label='Interpolated Data')
        ax_.set_title(f'Time Series for {station}')
        ax_.legend()
        value = None
    elif proximity_type == 'r2':
        value = r2_score(buoy_data, interpolated_data)
        sns.regplot(x=buoy_data, y=interpolated_data, ax=ax_)
        ax_.set_title(f'R2 Score for {station} with a score: {value:.2f}')      
    else:
        raise ValueError('Invalid proximity type')
    
    return value
    
    

In [10]:
def visualize_correlation_variables(stations: list, ds_buoy_filtered: xr.Dataset, variable: str, proximity: str):
       
    n_cols = 3
    n_rows = (len(stations) // n_cols) + (len(stations) % n_cols > 0)

    # Create a figure with subplots
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
    axes = axes.flatten()  # Flatten the axes array for easy indexing  
    similarity_values = []    
    for i, station in enumerate(stations):    
        
        
        buoy_data, interp_data = post_process_before_corr(ds_buoy_filtered, station, variable)
        value_similarity = calculate_proximity(proximity, buoy_data, interp_data, axes[i])
        similarity_values.append(value_similarity)
        
        
        
    # Hide any unused subplots
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()  # Adjust layout
    plt.show()
    
    return similarity_values
    

In [ ]:
year = 2019
stations = stations_dict[str(year)]
proximity = visualize_correlation_variables(stations, ds_buoy_filtered_dict[str(year)], 'u', 'r2') 